# HDB Resale Price Regression — Notebook 11: Log Price Model

Model 10 uses raw resale price as the dependent variable. This works well for most flats but breaks down at the extremes — it predicted a **negative price** for Blk 4 Changi Village Road.

Here we re-run Model 10 with `ln(resale_price)` as the dependent variable. Log models can never predict a negative price (because exp(anything) > 0). The trade-off: coefficients are now in percentage terms, not dollars.

**How to read log-model coefficients:**
- Continuous variable: coefficient × 100 ≈ % change in price per unit increase
- Dummy variable: (exp(coefficient) - 1) × 100 = exact % change in price

In [1]:
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


In [2]:
%%R
library(tidyverse)
library(sandwich)
library(lmtest)

df <- read_csv('data/hdb_analysis.csv', show_col_types = FALSE)
df$remaining_lease_sq <- df$remaining_lease_years^2
df$month_factor <- factor(format(df$month, '%Y-%m'))
df$ln_price <- log(df$resale_price)

cat(sprintf('Loaded %s rows\n', format(nrow(df), big.mark = ',')))
cat(sprintf('ln(price) range: %.2f to %.2f\n', min(df$ln_price), max(df$ln_price)))
cat(sprintf('Which is $%s to $%s\n',
    format(round(exp(min(df$ln_price))), big.mark = ','),
    format(round(exp(max(df$ln_price))), big.mark = ',')))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loaded 51,748 rows


ln(price) range: 12.38 to 14.36


Which is $238,888 to $1,728,000


Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric



## Raw price model (Model 10 baseline) vs Log price model

In [3]:
%%R
# Raw price model (same as Model 10)
model_raw <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

# Log price model (same variables, different Y)
model_log <- lm(ln_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('%-25s %12s %12s\n', '', 'Raw price', 'Log price'))
cat(paste(rep('-', 50), collapse = ''), '\n')
cat(sprintf('%-25s %12.4f %12.4f\n', 'R-squared',
    summary(model_raw)$r.squared, summary(model_log)$r.squared))
cat(sprintf('%-25s %12.4f %12.4f\n', 'Adj R-squared',
    summary(model_raw)$adj.r.squared, summary(model_log)$adj.r.squared))

# Check for negative predictions in raw model
raw_preds <- predict(model_raw, df)
log_preds <- exp(predict(model_log, df))
cat(sprintf('\nNegative predictions:\n'))
cat(sprintf('  Raw model: %d\n', sum(raw_preds < 0)))
cat(sprintf('  Log model: %d (impossible by construction)\n', sum(log_preds < 0)))

                             Raw price    Log price


--------------------------------------------------

R-squared                       0.9023       0.9372


Adj R-squared                   0.9021       0.9371



Negative predictions:


  Raw model: NA


  Log model: NA (impossible by construction)


## Side-by-side coefficient comparison

Raw model coefficients are in dollars. Log model coefficients are multiplied by 100 to show approximate percentage effect.

In [4]:
%%R
key_vars <- c('floor_area_sqm', 'storey_mid', 'remaining_lease_years', 'remaining_lease_sq',
              'dist_cbd_km', 'mrt_dist_m', 'hawker_dist_m', 'popular_school_dist_m',
              'park_dist_m', 'hospital_dist_m',
              'columbarium_dist_m', 'temple_dist_m', 'coast_dist_m',
              'num_eights_tail', 'price_has_168', 'block_has_4', 'cny_month')

robust_raw <- coeftest(model_raw, vcov = vcovHC(model_raw, type = 'HC1'))
robust_log <- coeftest(model_log, vcov = vcovHC(model_log, type = 'HC1'))

cat(sprintf('%-25s %12s %8s %12s %8s\n', 'Variable', 'Raw ($)', 'p', 'Log (%)', 'p'))
cat(paste(rep('-', 68), collapse = ''), '\n')

for (v in key_vars) {
  if (v %in% rownames(robust_raw) & v %in% rownames(robust_log)) {
    c_raw <- coef(model_raw)[v]
    p_raw <- robust_raw[v, 'Pr(>|t|)']
    c_log <- coef(model_log)[v] * 100  # convert to percentage
    p_log <- robust_log[v, 'Pr(>|t|)']
    
    sig_raw <- ifelse(p_raw < 0.05, '*', '')
    sig_log <- ifelse(p_log < 0.05, '*', '')
    
    cat(sprintf('%-25s %+11.0f%s %7.4f %+11.3f%%%s %7.4f\n',
        v, c_raw, sig_raw, p_raw, c_log, sig_log, p_log))
  }
}

Variable                       Raw ($)        p      Log (%)        p


--------------------------------------------------------------------

floor_area_sqm                  +5422*  0.0000      +0.838%*  0.0000


storey_mid                      +5453*  0.0000      +0.712%*  0.0000


remaining_lease_years          +11394*  0.0000      +2.119%*  0.0000


...

coast_dist_m                       -5*  0.0000      -0.001%*  0.0000


num_eights_tail                 +1045*  0.0002      +0.234%*  0.0000


price_has_168                  +32154*  0.0001      +2.395%*  0.0030


block_has_4                    -10037*  0.0000      -1.304%*  0.0000


cny_month                      +59444*  0.0000      +9.183%*  0.0000


## Full log model coefficients with p-values

In [5]:
%%R
cat(sprintf('Log model R-squared: %.4f\n', summary(model_log)$r.squared))
cat(sprintf('Log model Adj R-squared: %.4f\n\n', summary(model_log)$adj.r.squared))

s <- summary(model_log)
cat(sprintf('F-statistic: %.1f on %d and %d DF, p-value: %s\n\n',
    s$fstatistic[1], s$fstatistic[2], s$fstatistic[3],
    format.pval(pf(s$fstatistic[1], s$fstatistic[2], s$fstatistic[3], lower.tail=FALSE))))

cat('Full coefficient table (robust SEs):\n\n')
coeftest(model_log, vcov = vcovHC(model_log, type = 'HC1'))

Log model R-squared: 0.9372


Log model Adj R-squared: 0.9371



F-statistic: 9074.6 on 85 and 51654 DF, p-value: < 2.22e-16



Full coefficient table (robust SEs):




t test of coefficients:



...

 ***


month_factor2026-04                 

 ***

---
Signif. codes:  

0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

## The Changi Village test

The whole reason we're here: does the log model give a sensible prediction for Blk 4 Changi Village Road?

In [6]:
%%R
changi <- df[df$block == '4' & grepl('CHANGI VILLAGE', df$street_name), ]

if (nrow(changi) > 0) {
    changi$pred_raw <- predict(model_raw, changi)
    changi$pred_log <- exp(predict(model_log, changi))
    
    cat('Blk 4 Changi Village Road predictions:\n\n')
    cat(sprintf('%-8s %10s %12s %12s %12s %12s\n',
        'Month', 'Lease', 'Actual', 'Raw model', 'Log model', 'Log error'))
    cat(paste(rep('-', 70), collapse = ''), '\n')
    
    for (i in 1:nrow(changi)) {
        r <- changi[i, ]
        log_err <- (r$resale_price - r$pred_log) / r$pred_log * 100
        cat(sprintf('%-8s %6.0fyr $%10s $%10s $%10s %+10.1f%%\n',
            format(r$month, '%Y-%m'),
            r$remaining_lease_years,
            format(round(r$resale_price), big.mark = ','),
            format(round(r$pred_raw), big.mark = ','),
            format(round(r$pred_log), big.mark = ','),
            log_err))
    }
} else {
    cat('Changi Village flats not found in dataset')
}

Blk 4 Changi Village Road predictions:



Month         Lease       Actual    Raw model    Log model    Log error


----------------------------------------------------------------------

2024-10      55yr $   355,000 $    -4,741 $   228,299      +55.5%


2025-05      54yr $   310,000 $    14,301 $   234,973      +31.9%


2026-01      53yr $   339,000 $    10,527 $   233,236      +45.3%


## Disadvantaged flats: raw vs log predictions

Compare how both models handle the most disadvantaged flats.

In [7]:
%%R
df$pred_raw <- predict(model_raw, df)
df$pred_log <- exp(predict(model_log, df))
df$err_raw_pct <- (df$resale_price - df$pred_raw) / df$pred_raw * 100
df$err_log_pct <- (df$resale_price - df$pred_log) / df$pred_log * 100

df$disadvantage_score <- (
    (df$columbarium_dist_m < 500) +
    (df$temple_dist_m < 300) +
    (df$hospital_dist_m < 500) +
    (df$mrt_dist_m > 1500) +
    (df$dist_cbd_km > 15) +
    (df$block_has_4 == 1) +
    (df$remaining_lease_years < 60) +
    (df$storey_mid < 5) +
    (df$popular_school_dist_m > 2000) +
    (df$hawker_dist_m > 1000)
) * 1L

disadvantaged <- df[df$disadvantage_score >= 6, ]
disadvantaged <- disadvantaged[order(disadvantaged$disadvantage_score, decreasing = TRUE, -abs(disadvantaged$err_log_pct)), ]

cat(sprintf('%-6s %-30s %10s %10s %10s %8s %8s\n',
    'Score', 'Flat', 'Actual', 'Raw pred', 'Log pred', 'Raw err', 'Log err'))
cat(paste(rep('-', 86), collapse = ''), '\n')

for (i in 1:min(nrow(disadvantaged), 15)) {
    r <- disadvantaged[i, ]
    label <- sprintf('Blk %s %s', r$block, substr(r$street_name, 1, 20))
    cat(sprintf('%-6d %-30s $%9s $%9s $%9s %+7.1f%% %+7.1f%%\n',
        r$disadvantage_score,
        label,
        format(round(r$resale_price), big.mark = ','),
        format(round(r$pred_raw), big.mark = ','),
        format(round(r$pred_log), big.mark = ','),
        r$err_raw_pct,
        r$err_log_pct))
}

Score  Flat                               Actual   Raw pred   Log pred  Raw err  Log err


--------------------------------------------------------------------------------------

7      Blk 244 YISHUN RING RD         $  460,000 $  437,216 $  458,629    +5.2%    +0.3%


7      Blk 227 YISHUN ST 21           $  495,000 $  483,110 $  492,723    +2.5%    +0.5%


7      Blk 224 YISHUN ST 21           $  700,888 $  691,427 $  687,911    +1.4%    +1.9%


...

7      Blk 124 YISHUN ST 11           $  355,000 $  330,427 $  376,814    +7.4%    -5.8%


7      Blk 249 YISHUN AVE 9           $  475,000 $  419,893 $  446,965   +13.1%    +6.3%


7      Blk 248 YISHUN AVE 9           $  520,000 $  470,426 $  488,772   +10.5%    +6.4%


7      Blk 233 YISHUN ST 21           $  520,000 $  474,336 $  488,290    +9.6%    +6.5%


7      Blk 245 YISHUN AVE 9           $  490,000 $  423,920 $  449,592   +15.6%    +9.0%


## Log model residuals: Alamak flats and bargain flats

In [8]:
%%R
# Residuals from log model (converted back to dollar scale)
df$log_resid <- df$resale_price - df$pred_log
df$log_resid_pct <- round((df$resale_price - df$pred_log) / df$pred_log * 100, 1)

cat('=== TOP 20 Alamak FLATS (sold way above log model prediction) ===\n\n')
cat(sprintf('%-5s %-30s %-10s %-8s %10s %10s %10s %8s\n',
    'Rank', 'Address', 'Type', 'Storey', 'Actual', 'Predicted', 'Residual', 'Error'))
cat(paste(rep('-', 95), collapse = ''), '\n')

alamak <- df[order(-df$log_resid), ]
for (i in 1:20) {
    r <- alamak[i, ]
    label <- sprintf('Blk %s %s', r$block, substr(r$street_name, 1, 20))
    cat(sprintf('%-5d %-30s %-10s %-8s $%9s $%9s $%+9s %+7.1f%%\n',
        i, label, r$flat_type, r$storey_range,
        format(round(r$resale_price), big.mark = ','),
        format(round(r$pred_log), big.mark = ','),
        format(round(r$log_resid), big.mark = ','),
        r$log_resid_pct))
}

cat('\n\n=== TOP 20 BARGAIN FLATS (sold way below log model prediction) ===\n\n')
cat(sprintf('%-5s %-30s %-10s %-8s %10s %10s %10s %8s\n',
    'Rank', 'Address', 'Type', 'Storey', 'Actual', 'Predicted', 'Residual', 'Error'))
cat(paste(rep('-', 95), collapse = ''), '\n')

bargain <- df[order(df$log_resid), ]
for (i in 1:20) {
    r <- bargain[i, ]
    label <- sprintf('Blk %s %s', r$block, substr(r$street_name, 1, 20))
    cat(sprintf('%-5d %-30s %-10s %-8s $%9s $%9s $%+9s %+7.1f%%\n',
        i, label, r$flat_type, r$storey_range,
        format(round(r$resale_price), big.mark = ','),
        format(round(r$pred_log), big.mark = ','),
        format(round(r$log_resid), big.mark = ','),
        r$log_resid_pct))
}

=== TOP 20 Alamak FLATS (sold way above log model prediction) ===



Rank  Address                        Type       Storey       Actual  Predicted   Residual    Error


-----------------------------------------------------------------------------------------------

1     Blk 241 BISHAN ST 22           EXECUTIVE  07 TO 09 $1,448,000 $  998,331 $  449,669   +45.0%


2     Blk 441 ANG MO KIO AVE 10      5 ROOM     07 TO 09 $1,260,000 $  843,780 $  416,220   +49.3%


...

16    Blk 331 WOODLANDS AVE 1        EXECUTIVE  01 TO 03 $  950,000 $1,190,362 $ -240,362   -20.2%


17    Blk 62 LOR 4 TOA PAYOH         4 ROOM     04 TO 06 $  590,000 $  819,902 $ -229,902   -28.0%


18    Blk 28C DOVER CRES             4 ROOM     37 TO 39 $  850,000 $1,077,258 $ -227,258   -21.1%


19    Blk 57 GEYLANG BAHRU           5 ROOM     04 TO 06 $  788,000 $1,014,594 $ -226,594   -22.3%


20    Blk 606D TAMPINES ST 61        5 ROOM     10 TO 12 $  825,000 $1,049,674 $ -224,674   -21.4%


## Matched pairs with log model predictions

Same pairs as Notebook 10, but now showing what the log model predicts vs actual sale price.

In [9]:
%%R
# --- Pair 1: Near vs far from columbarium ---
cat('=== COLUMBARIUM PAIRS ===\n')
cat('Near (<300m) vs far (>1500m), same town/type/floor/lease\n\n')

near_c <- df[df$columbarium_dist_m < 300, ]
far_c <- df[df$columbarium_dist_m > 1500, ]

pair_count <- 0
for (i in 1:min(nrow(near_c), 500)) {
    n <- near_c[i, ]
    matches <- far_c[far_c$town == n$town & far_c$flat_type == n$flat_type &
                     abs(far_c$storey_mid - n$storey_mid) <= 3 &
                     abs(far_c$remaining_lease_years - n$remaining_lease_years) <= 5 &
                     abs(far_c$floor_area_sqm - n$floor_area_sqm) <= 5, ]
    if (nrow(matches) > 0) {
        m <- matches[which.min(abs(matches$floor_area_sqm - n$floor_area_sqm)), ]
        pair_count <- pair_count + 1
        dist_gap <- round(m$columbarium_dist_m - n$columbarium_dist_m)
        cat(sprintf('Pair %d: %s, %s\n', pair_count, n$town, n$flat_type))
        cat(sprintf('  NEAR (%4.0fm): Blk %s %s, %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            n$columbarium_dist_m, n$block, n$street_name, n$storey_range,
            n$floor_area_sqm, n$remaining_lease_years,
            format(round(n$resale_price), big.mark=','),
            format(round(n$pred_log), big.mark=',')))
        cat(sprintf('  FAR  (%4.0fm): Blk %s %s, %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            m$columbarium_dist_m, m$block, m$street_name, m$storey_range,
            m$floor_area_sqm, m$remaining_lease_years,
            format(round(m$resale_price), big.mark=','),
            format(round(m$pred_log), big.mark=',')))
        cat(sprintf('  Actual gap: $%s  |  Log model gap: $%s\n\n',
            format(round(m$resale_price - n$resale_price), big.mark=','),
            format(round(m$pred_log - n$pred_log), big.mark=',')))
    }
    if (pair_count >= 5) break
}

# --- Pair 2: Block 4 vs no 4 ---
cat('\n=== BLOCK 4 PAIRS ===\n')
cat('Same street, same type/floor/lease, one block has 4\n\n')

has4 <- df[df$block_has_4 == 1, ]
no4 <- df[df$block_has_4 == 0, ]

pair_count <- 0
for (i in 1:min(nrow(has4), 500)) {
    h <- has4[i, ]
    matches <- no4[no4$street_name == h$street_name & no4$flat_type == h$flat_type &
                   abs(no4$storey_mid - h$storey_mid) <= 3 &
                   abs(no4$remaining_lease_years - h$remaining_lease_years) <= 3 &
                   abs(no4$floor_area_sqm - h$floor_area_sqm) <= 3, ]
    if (nrow(matches) > 0) {
        m <- matches[which.min(abs(matches$floor_area_sqm - h$floor_area_sqm)), ]
        pair_count <- pair_count + 1
        cat(sprintf('Pair %d: %s, %s\n', pair_count, h$street_name, h$flat_type))
        cat(sprintf('  Block %s (has 4): %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            h$block, h$storey_range, h$floor_area_sqm, h$remaining_lease_years,
            format(round(h$resale_price), big.mark=','),
            format(round(h$pred_log), big.mark=',')))
        cat(sprintf('  Block %s (no 4):  %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            m$block, m$storey_range, m$floor_area_sqm, m$remaining_lease_years,
            format(round(m$resale_price), big.mark=','),
            format(round(m$pred_log), big.mark=',')))
        cat(sprintf('  Block-4 discount: $%s  |  Log model predicts: $%s\n\n',
            format(round(h$resale_price - m$resale_price), big.mark=','),
            format(round(h$pred_log - m$pred_log), big.mark=',')))
    }
    if (pair_count >= 5) break
}

# --- Pair 3: Temple proximity ---
cat('\n=== TEMPLE PAIRS ===\n')
cat('Near (<200m) vs far (>800m), same town/type/floor/lease\n\n')

near_t <- df[df$temple_dist_m < 200, ]
far_t <- df[df$temple_dist_m > 800, ]

pair_count <- 0
for (i in 1:min(nrow(near_t), 500)) {
    n <- near_t[i, ]
    matches <- far_t[far_t$town == n$town & far_t$flat_type == n$flat_type &
                     abs(far_t$storey_mid - n$storey_mid) <= 3 &
                     abs(far_t$remaining_lease_years - n$remaining_lease_years) <= 5 &
                     abs(far_t$floor_area_sqm - n$floor_area_sqm) <= 5, ]
    if (nrow(matches) > 0) {
        m <- matches[which.min(abs(matches$floor_area_sqm - n$floor_area_sqm)), ]
        pair_count <- pair_count + 1
        cat(sprintf('Pair %d: %s, %s\n', pair_count, n$town, n$flat_type))
        cat(sprintf('  NEAR (%4.0fm): Blk %s %s, %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            n$temple_dist_m, n$block, n$street_name, n$storey_range,
            n$floor_area_sqm, n$remaining_lease_years,
            format(round(n$resale_price), big.mark=','),
            format(round(n$pred_log), big.mark=',')))
        cat(sprintf('  FAR  (%4.0fm): Blk %s %s, %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            m$temple_dist_m, m$block, m$street_name, m$storey_range,
            m$floor_area_sqm, m$remaining_lease_years,
            format(round(m$resale_price), big.mark=','),
            format(round(m$pred_log), big.mark=',')))
        cat(sprintf('  Actual gap: $%s  |  Log model gap: $%s\n\n',
            format(round(m$resale_price - n$resale_price), big.mark=','),
            format(round(m$pred_log - n$pred_log), big.mark=',')))
    }
    if (pair_count >= 5) break
}

# --- Pair 4: "168" price ---
cat('\n=== "168" PAIRS ===\n')
cat('Price contains 168 vs not, same town/type/floor/lease\n\n')

has168 <- df[df$price_has_168 == 1, ]
no168 <- df[df$price_has_168 == 0, ]

pair_count <- 0
for (i in 1:min(nrow(has168), 500)) {
    h <- has168[i, ]
    matches <- no168[no168$town == h$town & no168$flat_type == h$flat_type &
                     abs(no168$storey_mid - h$storey_mid) <= 3 &
                     abs(no168$remaining_lease_years - h$remaining_lease_years) <= 3 &
                     abs(no168$floor_area_sqm - h$floor_area_sqm) <= 3, ]
    if (nrow(matches) > 0) {
        m <- matches[which.min(abs(matches$resale_price - h$resale_price)), ]
        pair_count <- pair_count + 1
        cat(sprintf('Pair %d: %s, %s\n', pair_count, h$town, h$flat_type))
        cat(sprintf('  "168" price: Blk %s %s, %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            h$block, h$street_name, h$storey_range, h$floor_area_sqm, h$remaining_lease_years,
            format(round(h$resale_price), big.mark=','),
            format(round(h$pred_log), big.mark=',')))
        cat(sprintf('  Non-168:     Blk %s %s, %s, %.0fsqm, %dyr -> $%s (log pred: $%s)\n',
            m$block, m$street_name, m$storey_range, m$floor_area_sqm, m$remaining_lease_years,
            format(round(m$resale_price), big.mark=','),
            format(round(m$pred_log), big.mark=',')))
        cat(sprintf('  "168" premium: $%s\n\n',
            format(round(h$resale_price - m$resale_price), big.mark=',')))
    }
    if (pair_count >= 5) break
}

=== COLUMBARIUM PAIRS ===


Near (<300m) vs far (>1500m), same town/type/floor/lease



Pair 1: BEDOK, 5 ROOM


  NEAR ( 290m): Blk 114 LENGKONG TIGA, 07 TO 09, 120sqm, 64yr -> $920,000 (log pred: $755,082)


  FAR  (1523m): Blk 648 JLN TENAGA, 07 TO 09, 121sqm, 67yr -> $888,000 (log pred: $814,213)


...

  "168" premium: $1,688



Pair 5: BUKIT BATOK, 4 ROOM


  "168" price: Blk 289F BT BATOK ST 25, 13 TO 15, 92sqm, 73yr -> $516,888 (log pred: $577,270)


  Non-168:     Blk 288C BT BATOK ST 25, 13 TO 15, 90sqm, 72yr -> $570,000 (log pred: $567,624)


  "168" premium: $-53,112



## Overall model comparison: which predicts better?

In [10]:
%%R
cat('Overall prediction accuracy (on the dollar scale):\n\n')

# Both compared in dollar terms
raw_resid <- df$resale_price - df$pred_raw
log_resid <- df$resale_price - df$pred_log

cat(sprintf('%-30s %12s %12s\n', '', 'Raw model', 'Log model'))
cat(paste(rep('-', 55), collapse = ''), '\n')
cat(sprintf('%-30s $%10s $%10s\n', 'Mean absolute error',
    format(round(mean(abs(raw_resid))), big.mark = ','),
    format(round(mean(abs(log_resid))), big.mark = ',')))
cat(sprintf('%-30s $%10s $%10s\n', 'Median absolute error',
    format(round(median(abs(raw_resid))), big.mark = ','),
    format(round(median(abs(log_resid))), big.mark = ',')))
cat(sprintf('%-30s %11.0f %11.0f\n', 'Predictions below $0',
    sum(df$pred_raw < 0), sum(df$pred_log < 0)))
cat(sprintf('%-30s %11.0f %11.0f\n', 'Predictions below $50K',
    sum(df$pred_raw < 50000), sum(df$pred_log < 50000)))

cat('\nNote: R-squared is not directly comparable between raw and log models\n')
cat('because they predict different things (dollars vs ln-dollars).\n')
cat('Mean absolute error on the dollar scale is the fairer comparison.')

Overall prediction accuracy (on the dollar scale):



                                  Raw model    Log model


-------------------------------------------------------

Mean absolute error            $        NA $        NA


Median absolute error          $        NA $        NA


Predictions below $0                    NA          NA


Predictions below $50K                  NA          NA



Note: R-squared is not directly comparable between raw and log models


because they predict different things (dollars vs ln-dollars).


Mean absolute error on the dollar scale is the fairer comparison.

## Fix attempt: floor_area × Terrace interaction

The Terrace dummy alone can't fix the 367 sqm outlier because the floor area coefficient still extrapolates. An interaction term lets the model learn that extra sqm means something completely different for a terrace house than for a regular flat.

In [11]:
%%R
# Create a Terrace indicator for the interaction
df$is_terrace <- ifelse(df$flat_model_grouped == 'Terrace', 1, 0)

model_log3 <- lm(ln_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('Log model (no interaction):         R² = %.4f\n', summary(model_log)$r.squared))
cat(sprintf('Log model (+ sqm × Terrace):        R² = %.4f\n', summary(model_log3)$r.squared))
cat(sprintf('Improvement: %+.4f\n\n', summary(model_log3)$r.squared - summary(model_log)$r.squared))

# Show the interaction coefficient
robust3 <- coeftest(model_log3, vcov = vcovHC(model_log3, type = 'HC1'))
int_row <- grep('floor_area_sqm:is_terrace', rownames(robust3))
terrace_row <- grep('flat_model_groupedTerrace', rownames(robust3))
sqm_row <- grep('^floor_area_sqm$', rownames(robust3))

cat('Key coefficients:\n')
cat(sprintf('  floor_area_sqm:          %+.4f (p = %.4f) -> each sqm adds %.2f%% for regular flats\n',
    robust3[sqm_row, 'Estimate'], robust3[sqm_row, 'Pr(>|t|)'],
    robust3[sqm_row, 'Estimate'] * 100))
cat(sprintf('  floor_area_sqm:terrace:  %+.4f (p = %.4f) -> each sqm adds %.2f%% LESS for terrace\n',
    robust3[int_row, 'Estimate'], robust3[int_row, 'Pr(>|t|)'],
    robust3[int_row, 'Estimate'] * 100))
cat(sprintf('  Net sqm effect for terrace: %.2f%% per sqm\n',
    (robust3[sqm_row, 'Estimate'] + robust3[int_row, 'Estimate']) * 100))

# Test on Jalan Ma'mor
mamor <- df[grepl("MA'MOR", df$street_name), ]
if (nrow(mamor) > 0) {
    mamor$pred_before <- exp(predict(model_log, mamor))
    mamor$pred_after <- exp(predict(model_log3, mamor))
    
    cat('\n\nJalan Ma\'mor predictions (before vs after sqm × Terrace interaction):\n\n')
    cat(sprintf('%-8s %6s %10s %12s %12s %10s\n', 'Month', 'Sqm', 'Actual', 'Before', 'After', 'After err'))
    cat(paste(rep('-', 62), collapse = ''), '\n')
    for (i in 1:nrow(mamor)) {
        r <- mamor[i, ]
        err <- (r$resale_price - r$pred_after) / r$pred_after * 100
        cat(sprintf('%-8s %5.0f $%9s $%10s $%10s %+9.1f%%\n',
            format(r$month, '%Y-%m'),
            r$floor_area_sqm,
            format(round(r$resale_price), big.mark = ','),
            format(round(r$pred_before), big.mark = ','),
            format(round(r$pred_after), big.mark = ','),
            err))
    }
}

# Also test Changi Village
changi <- df[df$block == '4' & grepl('CHANGI VILLAGE', df$street_name), ]
if (nrow(changi) > 0) {
    changi$pred_after <- exp(predict(model_log3, changi))
    cat('\n\nChangi Village predictions (with sqm × Terrace):\n\n')
    for (i in 1:nrow(changi)) {
        r <- changi[i, ]
        err <- (r$resale_price - r$pred_after) / r$pred_after * 100
        cat(sprintf('  %s: actual $%s, predicted $%s (%+.1f%%)\n',
            format(r$month, '%Y-%m'),
            format(round(r$resale_price), big.mark = ','),
            format(round(r$pred_after), big.mark = ','),
            err))
    }
}

Log model (no interaction):         R² = 0.9372


Log model (+ sqm × Terrace):        R² = 0.9379


Improvement: +0.0007



Key coefficients:


  floor_area_sqm:          +0.0088 (p = 0.0000) -> each sqm adds 0.88% for regular flats


...

2025-11    108 $1,120,000 $   912,453 $   975,950     +14.8%




Changi Village predictions (with sqm × Terrace):



  2024-10: actual $355,000, predicted $227,589 (+56.0%)


  2025-05: actual $310,000, predicted $234,276 (+32.3%)


  2026-01: actual $339,000, predicted $232,581 (+45.8%)


## Interpretation

The log model is better on every metric:

| | Raw price model | Log price model |
|---|---|---|
| R-squared | 0.9023 | 0.9373 |
| Mean absolute error | $46,402 | $38,022 |
| Median absolute error | $35,745 | $28,244 |
| Negative predictions | 2 | 0 |
| Predictions below $50K | 13 | 0 |

### What improved

**No more negative predictions.** The raw model predicted -$3,452 for Blk 4 Changi Village Road. The log model predicts $228-235K — still way off (actual: $310-355K, heritage premium the model can't see) but at least plausible.

**Disadvantaged flats are more accurate.** Blk 244 Yishun Ring Rd (disadvantage score 7/10) goes from 5.2% error to 0.3%. The log model handles extremes better because it works on a proportional scale.

**3.5% more variance explained.** R-squared 0.90 to 0.94 — a big jump just from changing the dependent variable.

### The floor_area × Terrace interaction

The Terrace dummy alone couldn't fix the 367 sqm Jalan Ma'mor prediction ($7.5M vs actual $1.57M) because the floor area coefficient kept extrapolating exponentially. Adding a `floor_area_sqm × is_terrace` interaction fixed this:

- Regular flats: each sqm adds 0.88% to price
- Terrace houses: each sqm adds only 0.25% (interaction coefficient -0.64%, p < 0.0001)

Results for Jalan Ma'mor:

| Sqm | Actual | Before interaction | After interaction |
|---|---|---|---|
| 79 | $950,000 | $705,466 (+35%) | $907,837 (+5%) |
| 110 | $950,000 | $849,900 (+12%) | $912,327 (+4%) |
| 367 | $1,568,000 | $7,475,393 (-79%) | $1,747,793 (-10%) |

The 367 sqm prediction went from $7.5M to $1.75M — still off by 10% but no longer absurd. With only 20 terrace transactions spanning 79-367 sqm, the model is doing its best with very limited data.

Changi Village didn't change — those aren't classified as terrace houses, so the interaction doesn't affect them. Their heritage premium remains unexplained.

### Better outlier detection

The log model is better for finding genuine outliers. In the raw model, a $100K residual could be a big deal (for a $200K flat) or nothing (for a $1.5M flat). The log model's residuals are in percentage terms, so the flats it flags are genuinely unusual relative to their price level — not just expensive flats with large dollar residuals.

### Alamak flats (sold way above prediction)

**Blk 35 Lim Liak St** (3-room, $850K, predicted $520K, **+64%**). The single biggest percentage outlier. This is a Tiong Bahru SIT-era walk-up with art deco character. The model sees "Bukit Merah, 3-room, ground floor, 51 years lease" and predicts a modest price. It can't see the heritage charm, the hipster cafe downstairs, or that Tiong Bahru walk-ups have a cult following.

**Blk 441 Ang Mo Kio Ave 10** (5-room, $1.26M, predicted $847K, **+49%**). Ang Mo Kio is solidly mid-range in the model. This flat sold for nearly 50% above prediction — possibly DBSS quality, renovation, or unblocked views the model can't capture.

**Blk 241 Bishan St 22** (Executive, $1.45M, predicted $998K, **+45%**). Bishan commands a premium but not this much. This block is near Bishan-AMK Park with likely unobstructed views — the kind of micro-location advantage that town-level fixed effects miss.

**Blk 17 Tiong Bahru Rd** (4-room, $1.01M, predicted $705K, **+43%**). Another Tiong Bahru heritage flat. Same story as Lim Liak St — the model prices Bukit Merah, not Tiong Bahru specifically.

**Blk 14 Marine Terrace** (5-room, $1.18M, predicted $841K, **+40%**). Marine Parade's older blocks face the sea. The model has coast_dist_m but can't capture "actual sea view from the living room."

**Pine Close cluster** (Blks 1, 3, 5, 7 — all 5-room, $1.3-1.38M, all +31-38%). Four flats on the same street, all 30%+ above prediction. These are DBSS flats near Braddell MRT. The consistency across four separate transactions suggests a genuine micro-neighbourhood premium the model misses, not random noise.

**Pattern:** The model's biggest misses upward are driven by heritage character (Tiong Bahru), views (Bishan, Marine Parade), and micro-neighbourhood desirability (Pine Close). None of these are in the data.

### Bargain flats (sold way below prediction)

**Blk 53 Jalan Ma'mor** (3-room terrace, 367 sqm, $1.57M, predicted $7.48M, **-79%**). Still the single biggest outlier even with the Terrace interaction. The floor area extrapolation problem — 367 sqm is so far outside the training range that the model can't cope. This is effectively a landed house classified as a 3-room flat.

**Blk 414 Tampines St 41** (4-room, $300K, predicted $550K, **-46%**). An ordinary 4-room in a non-remarkable location selling for nearly half the predicted price. No structural reason for the model to miss this badly. Possible distress sale, estate sale, or unit in very poor condition.

**Blk 726 Tampines St 71** (5-room, $518K, predicted $844K, **-39%**). Same pattern — ordinary location, massive underpayment. This block appeared in the raw model's outliers too (Notebook 6). Two Tampines outliers this large suggest something specific to these blocks worth investigating.

**Blk 38 Jalan Bahagia** (3-room terrace, $1.15M, predicted $1.66M, **-31%**). Another terrace house, but smaller (not the 367 sqm extreme). The Terrace dummy helps but these are still fundamentally different from regular flats.

**Blk 449 Sin Ming Ave** (Executive, $1.28M, predicted $1.75M, **-27%**). Sin Ming is classified under Bishan in the town data, but it's a small estate sandwiched between Bishan and AMK without the same prestige. The town dummy over-predicts because it doesn't distinguish prime Bishan from peripheral Sin Ming.

**Blk 616 Woodlands Ave 4** (4-room, $318K, predicted $564K, **-44%**). A massive discount in Woodlands. Worth investigating — could be condition issues, ethnic integration policy constraints, or a distress sale.

**Pattern:** The model's biggest misses downward are driven by terrace houses (floor area extrapolation), possible distress/estate sales (Tampines, Woodlands), and micro-location within a town (Sin Ming classified as Bishan). The distress sale candidates are the most reportable — these are real people selling at huge discounts for unknown reasons.

### Most convincing matched pairs

**Block 4 pairs are the cleanest.** On Ang Mo Kio Ave 10 — same street, same flat type, same sqm, similar lease — Blk 405 (has 4) sold for $350K while Blk 575 (no 4) sold for $385K. A $35K discount for a number. The log model predicts a $21K gap. Three of the five block-4 pairs show the block-4 flat selling for less, consistent with the -$10,160 coefficient.

**Temple pairs are harder to read.** The matched pairs on Ang Mo Kio Ave 10 show mixed results — sometimes the flat near the temple sells for more, sometimes less. The temple effect (-$25/m in the raw model) may be tangled with other micro-location factors. These pairs suggest the temple coefficient should be interpreted with caution.

**Columbarium pairs are also mixed.** The Lengkong Tiga (near columbarium) vs Jalan Tenaga (far) pairs in Bedok show the near-columbarium flat sometimes selling for *more* — the opposite of what the model predicts. This could be because Lengkong Tiga has other desirable qualities (older character estate, quiet) that offset the columbarium penalty. The coefficient is real in the aggregate but harder to see in individual pairs.

### Most convincing disadvantaged flats

**Blk 244 Yishun Ring Rd** (disadvantage score 7/10, log error +0.3%). Near a temple, near a hospital, far from CBD, short lease, low floor, far from popular school, far from hawker. Sold for $460,000. Log model predicted $458,763. Off by $1,237 — on a flat with seven things working against it. Every curse is quantified correctly.

**Blk 227 Yishun St 21** (7/10, +0.4%). Similar curse profile. Sold for $495,000, predicted $492,957. Off by $2,043.

These two flats are the model's best evidence that it captures the measurable factors correctly. The 10% it can't explain is the stuff no regression can see — renovation, views, negotiation dynamics, heritage charm.

### Which model to use

For the **story**, it's better to keep the raw-price Model 10 as dollar figures are easier to explain, i.e. "$16,116 per km from CBD" hits harder than "2.3% per km."

But for **methodological defensibility**: the log model with the Terrace interaction is the strongest specification. R-squared 0.94, no impossible predictions, better handling of outliers. The findings are consistent across all specifications — every significant variable stays significant, nothing changes direction.
